# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ali-Haider987/alihaider-flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Lane: Content Decline / Refresh Prioritization — same contract as `w03_data_contract.ipynb`

*(Skills loaded: `hunting-leakage-and-validating`, `querying-big-datasets`, and `flyrank/flyrank-data` per `skills/README.md`.)*

Unit of analysis: one content item, one client, for the mid-panel month **2026-03**, split at the 15th (features from days 1–15, outcome/label from days 16–31). This notebook builds a richer feature vector than the data contract's 5-feature minimum, adds categorical handling, and then goes hunting for leakage on purpose — including subtler leaks than the obvious one, and a privacy check.

In [1]:
# Setup: connect DuckDB to the hosted release (self-contained — this notebook doesn't assume
# w03_data_contract.ipynb has already been run in this session).
%pip -q install duckdb huggingface_hub

import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

def columns_of(table_sql):
    return [r[0] for r in con.sql(f'DESCRIBE SELECT * FROM {table_sql} LIMIT 0').fetchall()]

def resolve_col(table_sql, candidates):
    cols = columns_of(table_sql)
    for c in candidates:
        if c in cols:
            return c
    raise ValueError(f'None of {candidates} found. Actual columns: {cols}')

print('fact_daily columns: ', columns_of(TABLES['fact_daily']))
print('dim_content columns:', columns_of(TABLES['dim_content']))
print('dim_clients columns:', columns_of(TABLES['dim_clients']))


Paste your Hugging Face READ token (hf_...): ··········
fact_daily columns:  ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
dim_content columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_c

In [2]:
# Base numeric features from the first half of March (pre-decision window), plus the
# second-half impression total we need ONLY to build the label — never as a feature.
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks      ELSE 0 END) AS clk_first_half,
           AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END)        AS pos_first_half,
           SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY 1, 2
    HAVING imp_first_half >= 20
""").df()

# Query-mix signals (content-level aggregates, repeated per query row -> ANY_VALUE + GROUP BY)
q_count_col = resolve_col(TABLES['fact_query_90d'], ['content_visible_query_count'])
q_rare_col  = resolve_col(TABLES['fact_query_90d'], ['rare_impressions_share'])
q_anon_col  = resolve_col(TABLES['fact_query_90d'], ['anonymized_impressions_share'])
qsig = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE({q_count_col}) AS visible_queries,
           ANY_VALUE({q_rare_col})  AS rare_share,
           ANY_VALUE({q_anon_col})  AS anon_share
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

# Static content attributes (numeric + categorical)
content_key = resolve_col(TABLES['dim_content'], ['content_hash_id'])
wc_col      = resolve_col(TABLES['dim_content'], ['word_count', 'content_word_count', 'wordcount'])
# content_type may or may not exist under that exact name -- probe for it instead of assuming.
content_cols = columns_of(TABLES['dim_content'])
type_candidates = [c for c in content_cols if 'type' in c.lower() or 'intent' in c.lower()]
type_col = type_candidates[0] if type_candidates else None
select_extra = f", {type_col} AS content_type" if type_col else ""
content_attrs = con.sql(f"""
    SELECT {content_key} AS content_hash_id, {wc_col} AS word_count {select_extra}
    FROM {TABLES['dim_content']}
""").df()
print('Detected categorical content column:', type_col)

# Client-level availability flag (categorical/boolean context, not a raw ID)
clients = con.sql(f"""
    SELECT client_hash_id, (ga4_data_start IS NOT NULL) AS has_ga4_export
    FROM {TABLES['dim_clients']}
""").df()

data = (feat
        .merge(qsig, on='content_hash_id', how='left')
        .merge(content_attrs, on='content_hash_id', how='left')
        .merge(clients, on='client_hash_id', how='left'))
print(f'{len(data):,} rows before missing-value handling')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Detected categorical content column: content_type
109,592 rows before missing-value handling


,client_hash_id,content_hash_id,imp_first_half,clk_first_half,pos_first_half,imp_second_half,visible_queries,rare_share,anon_share,word_count,content_type,has_ga4_export
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,6.327311,2350.0,57.0,0.067073,0.783492,2123,keyword article,True
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,3.906852,208.0,NaN,NaN,NaN,<NA>,keyword article,True
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,6.473735,1925.0,43.0,0.040734,0.780882,2546,keyword article,True
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,7.259861,2504.0,12.0,0.019587,0.960261,2330,keyword article,True
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,1.0,3.860842,189.0,2.0,0.038851,0.925676,<NA>,keyword article,True


In [3]:
# --- Missing-value + categorical handling ---
# Numeric query-mix signals: missing means "no 90d query record for this content item yet" --
# genuinely unknown, not zero. Fill with 0 but keep a companion flag so the model can tell
# 'observed zero' apart from 'no record'.
for col in ['visible_queries', 'rare_share', 'anon_share']:
    data[f'{col}_missing'] = data[col].isna().astype(int)
    data[col] = data[col].fillna(0)

# word_count: missing means the catalog entry is incomplete -- fill with the dataset median,
# flagged, rather than 0 (0 words would misleadingly look like the shortest possible page).
data['word_count_missing'] = data['word_count'].isna().astype(int)
data['word_count'] = data['word_count'].fillna(data['word_count'].median())

# has_ga4_export: boolean -> int, no missing expected (dim_clients is small and complete)
data['has_ga4_export'] = data['has_ga4_export'].fillna(False).astype(int)

# content_type (if found): categorical -> one-hot, with an explicit 'unknown' bucket
if 'content_type' in data.columns:
    data['content_type'] = data['content_type'].fillna('unknown')
    dummies = pd.get_dummies(data['content_type'], prefix='content_type')
    data = pd.concat([data, dummies], axis=1)
    categorical_feature_cols = list(dummies.columns)
else:
    categorical_feature_cols = []

numeric_feature_cols = ['imp_first_half', 'clk_first_half', 'pos_first_half',
                         'visible_queries', 'rare_share', 'anon_share', 'word_count',
                         'has_ga4_export']
missing_flag_cols = ['visible_queries_missing', 'rare_share_missing', 'anon_share_missing', 'word_count_missing']

honest_feature_cols = numeric_feature_cols + missing_flag_cols + categorical_feature_cols
print(f'{len(honest_feature_cols)} honest features:', honest_feature_cols)
data[honest_feature_cols].head()


15 honest features: ['imp_first_half', 'clk_first_half', 'pos_first_half', 'visible_queries', 'rare_share', 'anon_share', 'word_count', 'has_ga4_export', 'visible_queries_missing', 'rare_share_missing', 'anon_share_missing', 'word_count_missing', 'content_type_comparison article', 'content_type_feedly article', 'content_type_keyword article']


,imp_first_half,clk_first_half,pos_first_half,visible_queries,rare_share,anon_share,word_count,has_ga4_export,visible_queries_missing,rare_share_missing,anon_share_missing,word_count_missing,content_type_comparison article,content_type_feedly article,content_type_keyword article
0,4173.0,6.0,6.327311,57.0,0.067073,0.783492,2123,1,0,0,0,0,False,False,True
1,245.0,0.0,3.906852,0.0,0.000000,0.000000,2767,1,1,1,1,1,False,False,True
2,3705.0,3.0,6.473735,43.0,0.040734,0.780882,2546,1,0,0,0,0,False,False,True
3,2440.0,8.0,7.259861,12.0,0.019587,0.960261,2330,1,0,0,0,0,False,False,True
4,240.0,1.0,3.860842,2.0,0.038851,0.925676,2767,1,0,0,0,1,False,False,True


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

| Feature | Meaning | Missing handling | Categorical? | Available at decision moment (2026-03-15)? |
|---|---|---|---|---|
| `imp_first_half` | GSC impressions, Mar 1–15 | dropped below floor 20 (see `HAVING`) | no | **Yes** — closed historical window |
| `clk_first_half` | GSC clicks, Mar 1–15 | same floor | no | **Yes** |
| `pos_first_half` | avg GSC position, Mar 1–15 | none observed; would fill with group median if it occurred | no | **Yes** |
| `visible_queries` | trailing-90d distinct visible query count | filled 0 + `_missing` flag | no | **Yes** — 90d snapshot predates the split, built independently of the March outcome |
| `rare_share` / `anon_share` | trailing-90d share of rare/anonymized query impressions | filled 0 + `_missing` flag | no | **Yes** — same reasoning |
| `word_count` | static content length | filled with dataset median + `_missing` flag | no | **Yes** — set at publish/last-edit time, no dependency on any performance window |
| `has_ga4_export` | whether the client has a GA4 BigQuery export configured | none (small, complete table) | boolean (already 0/1) | **Yes** — a standing account-configuration fact, not a March event |
| `content_type_*` (one-hot, if present) | content category / intent bucket | `unknown` bucket added before one-hot | **yes** | **Yes** — static catalog attribute |
| `*_missing` flags | "this value had to be imputed" | n/a (they ARE the missingness signal) | boolean | **Yes** — computed from the same pre-decision data |

**Why `_missing` flags matter here:** filling a query-mix signal with 0 is ambiguous — it could mean "this page truly has almost no visible queries" or "we have no 90-day record for it at all." Keeping a companion flag lets the model (and a human reviewer) tell those apart instead of silently treating unknown-and-zero the same.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### The leakage hunt

I attack my own feature set three ways: an obvious label-derived leak (as in `w03_data_contract.ipynb`), a **subtler** leak that mixes a bit of the outcome window into a "feature" through a windowing bug, and a **privacy** check that the feature frame carries no raw identifiers or query text.

In [4]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

data['is_declining'] = (data['imp_second_half'] < 0.8 * data['imp_first_half']).astype(int)
model_data = data.dropna(subset=honest_feature_cols + ['is_declining']).copy()

def score(feature_cols, df, label_col='is_declining', seed=42):
    X = df[feature_cols].astype(float)
    y = df[label_col]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    scaler = StandardScaler().fit(X_tr)
    clf = LogisticRegression(max_iter=1000, class_weight='balanced').fit(scaler.transform(X_tr), y_tr)
    return roc_auc_score(y_te, clf.predict_proba(scaler.transform(X_te))[:, 1])

honest_auc = score(honest_feature_cols, model_data)
print(f'Honest AUC (all pre-decision features): {honest_auc:.3f}')


Honest AUC (all pre-decision features): 0.697


**Leak 1 — the obvious one:** feed in the exact ratio the label thresholds on.

In [5]:
leak1 = model_data.copy()
leak1['second_to_first_ratio'] = leak1['imp_second_half'] / leak1['imp_first_half']
leak1_auc = score(honest_feature_cols + ['second_to_first_ratio'], leak1)
print(f'Leak 1 AUC (adds second_to_first_ratio): {leak1_auc:.3f}  <- jumps toward 1.0, for the wrong reason')


Leak 1 AUC (adds second_to_first_ratio): 1.000  <- jumps toward 1.0, for the wrong reason


**Leak 2 — the subtle one:** a windowing bug. Suppose the "first half" aggregation was accidentally written as `report_date <= DATE '2026-03-16'` instead of `'2026-03-15'` — one extra day that actually belongs to the outcome window. This kind of off-by-one is exactly the sort of leak that doesn't announce itself; it just quietly inflates the score.

In [6]:
buggy = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date <= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_first_half_buggy
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY 1, 2
""").df()

leak2 = model_data.merge(buggy, on=['client_hash_id', 'content_hash_id'], how='left')
leak2['imp_first_half_buggy'] = leak2['imp_first_half_buggy'].fillna(leak2['imp_first_half'])
leak2_features = [c if c != 'imp_first_half' else 'imp_first_half_buggy' for c in honest_feature_cols]
leak2_auc = score(leak2_features, leak2)
print(f'Honest AUC:                         {honest_auc:.3f}')
print(f'Leak 2 AUC (one-day window overlap): {leak2_auc:.3f}')
print('Even a single day of outcome-window overlap measurably moves the score --',
      'this is why window boundaries in the data contract get their own verification query.')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest AUC:                         0.697
Leak 2 AUC (one-day window overlap): 0.692
Even a single day of outcome-window overlap measurably moves the score -- this is why window boundaries in the data contract get their own verification query.


**Remove both leaks, keep the honest number.**

In [7]:
final_feature_cols = honest_feature_cols
print(f'Final feature set kept for reporting ({len(final_feature_cols)} features): {final_feature_cols}')
print(f'Final, honest AUC: {honest_auc:.3f}')


Final feature set kept for reporting (15 features): ['imp_first_half', 'clk_first_half', 'pos_first_half', 'visible_queries', 'rare_share', 'anon_share', 'word_count', 'has_ga4_export', 'visible_queries_missing', 'rare_share_missing', 'anon_share_missing', 'word_count_missing', 'content_type_comparison article', 'content_type_feedly article', 'content_type_keyword article']
Final, honest AUC: 0.697


**Privacy check:** confirm the feature frame carries no raw identifiers or free text — only hashed keys and aggregates, as `DATA_USE.md` requires.

In [8]:
import re

suspicious_patterns = ['query_text', 'query_string', 'url', 'domain', 'client_name', 'title', 'keyword_text']
flagged = [c for c in model_data.columns if any(p in c.lower() for p in suspicious_patterns)]
print('Columns flagged as potentially identifying/free-text:', flagged if flagged else 'none found')

# Spot-check the hash-key columns actually look like opaque hashes, not readable strings.
sample_ids = model_data[['client_hash_id', 'content_hash_id']].head(3)
print('\nSample key values (should be opaque hashes, not names/URLs):')
print(sample_ids.to_string(index=False))

assert len(flagged) == 0, 'Found a potentially identifying column -- drop it before publishing anything.'
print('\nPrivacy check passed: no raw identifiers or free text in the feature frame.')


Columns flagged as potentially identifying/free-text: none found

Sample key values (should be opaque hashes, not names/URLs):
         client_hash_id          content_hash_id
client_73cda7b4e4f265ea content_7a105f548d9c6916
client_73cda7b4e4f265ea content_a3ea9792f793ec72
client_73cda7b4e4f265ea content_36c36abc7650d7af

Privacy check passed: no raw identifiers or free text in the feature frame.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### What I excluded, and why

| Excluded | Why |
|---|---|
| `imp_second_half`, `second_to_first_ratio`, and any other second-half aggregate | Anything from the outcome window is the answer key by definition — that's the whole leakage lesson (Leak 1). |
| `imp_first_half_buggy` (off-by-one window) | Demonstrates that even a small window-boundary bug is a leak, not just deliberate label-derived features (Leak 2). |
| FlyRank's own `trend_*`, `health_score`, `recommended_action`, `action_type` | Named explicitly on FlyRank's `internship-lanes` dataset card as leakage-risk context, not default features — excluded on principle, even though this notebook never queries that table. |
| Raw GA4 event-level rows | Finer grain than this lane's unit of analysis (content-item-month); not needed for a GSC-driven decline signal. |
| Individual query strings (only `fact_query_90d`'s pre-aggregated shares/counts are used) | Query text is long-tail and can be near-identifying; the aggregates give the needed signal without touching raw text. |
| `client_hash_id` / `content_hash_id` as model *features* | Kept only as join/index keys, never passed into `X` — a model that partly memorizes IDs isn't learning a generalizable pattern, and IDs are exactly the kind of identifier `DATA_USE.md` says to keep out of anything published. |
| The sealed `fact_content_daily_performance_sample` (2026-06) table | Reserved as a held-out test month; using it to shape features or labels now would be the same past→future leakage this whole notebook is about avoiding. |

In [9]:
# Back it with a check: confirm none of the excluded/leak columns made it into the final matrix.
excluded_names = ['imp_second_half', 'second_to_first_ratio', 'imp_first_half_buggy',
                   'trend_pct', 'trend_direction', 'health_score', 'recommended_action', 'action_type']
leaked_into_final = [c for c in final_feature_cols if c in excluded_names]
print('Excluded columns that leaked into the final feature set (should be empty):', leaked_into_final)
assert len(leaked_into_final) == 0, 'An excluded/leak column made it into the final feature set!'
print('Confirmed: final feature set contains none of the excluded columns.')


Excluded columns that leaked into the final feature set (should be empty): []
Confirmed: final feature set contains none of the excluded columns.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.